# Market-data ETL — task runner

Pick one or more **tasks** and edit their **config section** below, then **Run All**.

**Tasks available**
- `md_snapshot_zip_to_parquet` — decompress daily snapshot zips into per-ticker parquet
  (`<OUT>/<yyyy>/<YYYYMMDD>/<ticker>.parquet`)
- `md_trade_zip_to_parquet` — decompress daily trade zips into per-ticker parquet
  (`<OUT>/<yyyy>/<YYYYMMDD>/<ticker>.parquet`)

Each task has its own section in the `TASKS` config dict; set `RUN_TASKS` to the list
of tasks to run (in order). The actual work runs in this repo's dedicated venv (`.venv`)
via the tested `tasks/<task>.py` script, so it always uses the right pandas/pyarrow no
matter which kernel this notebook is running on.

> Safety: `DRY_RUN = True` by default — it only lists the days that *would* run.
> Set `DRY_RUN = False` to actually convert. Re-running a day overwrites its
> existing parquet files.

In [ ]:
# ============================ CONFIG — edit me ============================
from pathlib import Path

# --- Shared settings — apply to every task in RUN_TASKS ------------------
RUN_TASKS = [                            # tasks to run, in order (see TASKS below)
    "md_snapshot_zip_to_parquet",
    "md_trade_zip_to_parquet",
]
MARKET  = "hkg"          # market code           (--market)
ASSET   = "eq"           # asset class           (--asset; eq = equity)
START   = "20250101"     # first day YYYYMMDD, or None = earliest available
END     = "20250131"     # last  day YYYYMMDD, or None = latest available
WORKERS = None           # None = script default (min(20, cpus)); or set an int
DRY_RUN = True           # True = just list the days; False = actually convert

REPO    = Path.cwd()                                  # this repo
VENV_PY = str(REPO / ".venv" / "bin" / "python")

# --- Per-task config — one section per task ------------------------------
TASKS = {
    # md_snapshot_zip_to_parquet: daily snapshot zips -> per-ticker parquet
    "md_snapshot_zip_to_parquet": {
        "script": str(REPO / "tasks" / "md_snapshot_zip_to_parquet.py"),
        "src":    "/home/wangfc/tmp_data",  # root of <YYYYMM>/<YYYYMMDD>.zip files
        "out":    None,      # None = derive from market/asset (.../md_snapshot/raw)
    },
    # md_trade_zip_to_parquet: daily trade zips -> per-ticker parquet
    "md_trade_zip_to_parquet": {
        "script": str(REPO / "tasks" / "md_trade_zip_to_parquet.py"),
        "src":    "/home/wangfc/md_trade",  # root of <YYYYMM>/<YYYYMMDD>.zip files
        "out":    None,      # None = derive from market/asset (.../md_trade/raw)
    },
}
# =========================================================================

assert RUN_TASKS, "RUN_TASKS is empty — list at least one task to run."
for _t in RUN_TASKS:
    assert _t in TASKS, f"Unknown task: {_t!r} (known: {sorted(TASKS)})"

In [ ]:
# Discover the available days for one task and apply the shared date range.
from pathlib import Path

def discover_days(cfg):
    """Return (available, lo, hi, days) for a task config.

    Globs <src>/*.zip and <src>/*/*.zip (per-month folders), extracts the
    YYYYMMDD stems, and filters to the inclusive shared START..END range
    (None on either side = earliest/latest available).
    """
    src = cfg["src"]
    available = sorted({p.stem for p in Path(src).glob("*.zip")}
                       | {p.stem for p in Path(src).glob("*/*.zip")})
    assert available, f"No *.zip found under {src}"

    lo = START or available[0]
    hi = END   or available[-1]
    days = [d for d in available if lo <= d <= hi]
    return available, lo, hi, days

In [ ]:
# Run each selected task over its chosen days, streaming progress live.
import subprocess

def run_task(task):
    cfg = TASKS[task]
    available, lo, hi, days = discover_days(cfg)

    print("=" * 72)
    print(f"Task          : {task}")
    print(f"Source        : {cfg['src']}")
    print(f"Available zips: {len(available)}  ({available[0]} .. {available[-1]})")
    print(f"Range         : {lo} .. {hi}")
    print(f"Days selected : {len(days)}")
    for d in days:
        print("   ", d)
    if not days:
        print("No days match the selected range — skipping.")
        return

    cmd = [VENV_PY, cfg["script"],
           "--market", MARKET, "--asset", ASSET,
           "--date-range", f"{lo}:{hi}", "--src", cfg["src"]]
    if cfg["out"]:
        cmd += ["--out", cfg["out"]]
    if WORKERS:
        cmd += ["--workers", str(WORKERS)]

    print("\nCommand:\n ", " ".join(cmd), "\n")

    if DRY_RUN:
        print("DRY_RUN = True — nothing executed. Set DRY_RUN = False to convert.")
        return

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print(f"Exit code: {rc}", "OK" if rc == 0 else "FAILURES — see output above")


for _task in RUN_TASKS:
    run_task(_task)